# Notebook 2 - Làm sạch dữ liệu, pipeline và feature (Bước 4 - 5)

Notebook này đọc ba tập `train`, `val`, `test` do Notebook 1 tạo ra (đủ 151 cột gốc) và làm tiếp:

4. Làm sạch dữ liệu và tạo **pipeline** học từ `train` rồi áp dụng sang `val`, `test`
5. Chọn và lọc bỏ các cột không phù hợp (mỗi quyết định có bằng chứng do code tính), rồi tạo feature mới

**Quy tắc:** mọi thứ *học một con số từ dữ liệu* (trung vị, độ lệch chuẩn, danh sách giá trị của biến phân loại, các ngưỡng) chỉ được tính trên `train`. `val` và `test` chỉ được *áp dụng*, không được dùng để học. Mỗi hàm một ô code, mỗi kết quả một ô code; mọi con số trong nhận xét đều do code in ra.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [ ]:
SEED = 42
ROOT = Path("..")
DATA_DIR = ROOT / "data/processed"
ART_DIR = ROOT / "artifacts"
FIG_DIR = ART_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "val.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")
data_dictionary = pd.read_csv(DATA_DIR / "data_dictionary.csv", index_col="column")

In [ ]:
print("train:", train.shape, "| val:", val.shape, "| test:", test.shape)

In [ ]:
assert train["issue_date"].max() < val["issue_date"].min()
assert val["issue_date"].max() < test["issue_date"].min()
print("Kiểm tra lại thứ tự thời gian: train < val < test.")

In [ ]:
ALL_COLUMNS = list(data_dictionary.index)
print("Số cột gốc theo từ điển:", len(ALL_COLUMNS))
print("Các cột phụ do Notebook 1 thêm:", [c for c in train.columns if c not in ALL_COLUMNS])

---
# Bước 4 - Làm sạch dữ liệu và tạo pipeline

**Tham khảo cách làm của người khác** (nêu tên và kỹ thuật theo tên/mô tả project; chưa đối chiếu nội dung gốc, mọi điều rút ra cho dữ liệu của ta đều do code bên dưới kiểm chứng):
- Tanish-Srivastava, `credit-risk-scorecard` - https://github.com/Tanish-Srivastava/credit-risk-scorecard : dùng danh sách cột được phép (whitelist) lúc nộp đơn.
- Arturo-GA, `lendingclub-default-risk` - https://github.com/Arturo-GA/lendingclub-default-risk : whitelist và kiểm tra tự động không để cột bị cấm lọt vào mô hình.
- vaibhavkev, `credit-risk` - https://github.com/vaibhavkev/credit-risk : whitelist cột và có test kiểm tra rò rỉ.

## 4.1 Tìm các giá trị bất thường trong dữ liệu
Ta chỉ **nhìn** các giá trị bất thường trên `train` để đặt ngưỡng. Hàm sau đếm số dòng vi phạm từng quy tắc logic.

In [ ]:
def parse_month(series):
    return pd.to_datetime(series, format="%b-%Y")

In [ ]:
def logic_checks(df):
    first_credit = parse_month(df["earliest_cr_line"])
    checks = {
        "annual_inc <= 0": df["annual_inc"] <= 0,
        "dti < 0": df["dti"] < 0,
        "dti > 100": df["dti"] > 100,
        "revol_util > 100": df["revol_util"] > 100,
        "fico_range_low > fico_range_high": df["fico_range_low"] > df["fico_range_high"],
        "tài khoản tín dụng đầu tiên mở sau ngày giải ngân": first_credit > df["issue_date"],
        "loan_amnt <= 0": df["loan_amnt"] <= 0,
    }
    return pd.Series({name: int(mask.sum()) for name, mask in checks.items()})

**Bảng 1.** Số dòng vi phạm từng quy tắc logic, trên cả ba tập (chỉ dùng để quan sát).

In [ ]:
checks_before = pd.DataFrame({"train": logic_checks(train), "val": logic_checks(val), "test": logic_checks(test)})
checks_before["id trùng (train)"] = ""
checks_before

In [ ]:
print("Số id trùng lặp trong train:", train["id"].duplicated().sum(),
      "| trong val:", val["id"].duplicated().sum(), "| trong test:", test["id"].duplicated().sum())
print("Số id xuất hiện ở nhiều hơn một tập:", len(set(train["id"]) & set(val["id"])) + len(set(train["id"]) & set(test["id"])) + len(set(val["id"]) & set(test["id"])))

Để đặt ngưỡng, ta xem phân phối ba cột hay có giá trị cực đoan, **chỉ trên train**.

**Bảng 2.** Phân vị của `dti`, `annual_inc`, `revol_util` trong train.

In [ ]:
train[["dti", "annual_inc", "revol_util"]].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).T

In [ ]:
n_dti_bad = ((train["dti"] < 0) | (train["dti"] > 100)).sum()
n_income_bad = (train["annual_inc"] <= 0).sum()
print(f"Số dòng train có dti ngoài khoảng 0-100: {n_dti_bad} ({n_dti_bad / len(train):.4%}).")
print(f"Số dòng train có annual_inc <= 0: {n_income_bad} ({n_income_bad / len(train):.4%}).")
print(f"revol_util: phân vị 99.9% là {train['revol_util'].quantile(0.999):.1f}, lớn nhất là {train['revol_util'].max():.1f}.")
print("Hành động -> dti là tỷ lệ phần trăm nên giá trị ngoài 0-100 coi là không hợp lệ; thu nhập <= 0 không hợp lệ; "
      "revol_util vượt 100 là có thể (dùng quá hạn mức) nên chỉ chặn trên ở phân vị 99.9% của train.")

## 4.2 Các hàm làm sạch
Mỗi quy tắc là một quy tắc **cố định theo từng dòng** (cùng một quy tắc cho cả ba tập, không học gì từ `val` hay `test`). Hai ngưỡng được cố định từ `train`:

In [ ]:
DTI_MAX = 100
REVOL_UTIL_CAP = round(float(train["revol_util"].quantile(0.999)), 1)
print("DTI_MAX =", DTI_MAX, "| REVOL_UTIL_CAP (phân vị 99.9% của train) =", REVOL_UTIL_CAP)

In [ ]:
def fix_invalid_values(df):
    df = df.copy()
    df.loc[df["annual_inc"] <= 0, "annual_inc"] = np.nan
    df.loc[(df["dti"] < 0) | (df["dti"] > DTI_MAX), "dti"] = np.nan
    df["revol_util"] = df["revol_util"].clip(upper=REVOL_UTIL_CAP)
    return df

In [ ]:
def drop_duplicate_ids(df):
    return df.drop_duplicates(subset="id")

In [ ]:
train_c = drop_duplicate_ids(fix_invalid_values(train))
val_c = drop_duplicate_ids(fix_invalid_values(val))
test_c = drop_duplicate_ids(fix_invalid_values(test))
print("Số dòng sau làm sạch - train:", len(train_c), "| val:", len(val_c), "| test:", len(test_c))

**Bảng 3.** Chạy lại các quy tắc logic sau khi làm sạch (các quy tắc đã sửa phải về 0 ở cả ba tập).

In [ ]:
checks_after = pd.DataFrame({"train": logic_checks(train_c), "val": logic_checks(val_c), "test": logic_checks(test_c)})
checks_after

In [ ]:
fixed_rules = ["annual_inc <= 0", "dti < 0", "dti > 100"]
assert (checks_after.loc[fixed_rules] == 0).all().all()
print("Các quy tắc đã sửa đều về 0 ở cả ba tập.")
still = checks_after[(checks_after > 0).any(axis=1)]
print("Quy tắc còn dòng vi phạm (không sửa vì là giá trị có thể xảy ra hoặc không bất thường):", still.index.tolist())

## 4.3 Giá trị thiếu
**Bảng 4.** 25 cột có tỷ lệ thiếu cao nhất trong `train`, cùng tỷ lệ thiếu ở `val` và `test` để thấy dữ liệu thiếu có ổn định theo thời gian không.

In [ ]:
missing_table = pd.DataFrame({
    "train_%": train_c[ALL_COLUMNS].isna().mean() * 100,
    "val_%": val_c[ALL_COLUMNS].isna().mean() * 100,
    "test_%": test_c[ALL_COLUMNS].isna().mean() * 100,
})
missing_table.sort_values("train_%", ascending=False).head(25)

In [ ]:
n_all_empty = (missing_table["train_%"] == 100).sum()
n_some = ((missing_table["train_%"] > 0) & (missing_table["train_%"] < 100)).sum()
shift = (missing_table["test_%"] - missing_table["train_%"]).abs().sort_values(ascending=False)
print(f"Cột trống hoàn toàn trong train: {n_all_empty}. Cột thiếu một phần: {n_some}. Cột không thiếu: {(missing_table['train_%'] == 0).sum()}.")
print(f"Cột có tỷ lệ thiếu thay đổi nhiều nhất giữa train và test: {shift.index[0]} (lệch {shift.iloc[0]:.1f} điểm %).")
print("Hành động -> việc bỏ cột thiếu nhiều sẽ được quyết định ở Bước 5 bằng ngưỡng tính trên train; "
      "việc điền giá trị thiếu (cần học trung vị) nằm trong pipeline ở mục 4.4.")

## 4.4 Pipeline: học từ train, áp dụng sang val và test
Pipeline xử lý hai loại cột:
- cột **số**: điền giá trị thiếu bằng trung vị rồi chuẩn hóa (trừ trung bình, chia độ lệch chuẩn);
- cột **phân loại** (chữ): mã hóa one-hot, với giá trị lạ chưa từng thấy ở train thì bỏ qua (`handle_unknown="ignore"`).

Trung vị, trung bình, độ lệch chuẩn và danh sách giá trị đều là các con số **học từ dữ liệu**, nên phải `fit` chỉ trên `train`. Danh sách cột số/cột phân loại được truyền vào như tham số vì sẽ do Bước 5 quyết định.

In [ ]:
def make_preprocessor(numeric_cols, categorical_cols):
    numeric_part = Pipeline([("impute", SimpleImputer(strategy="median")),
                             ("scale", StandardScaler())])
    categorical_part = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return ColumnTransformer([("num", numeric_part, numeric_cols),
                              ("cat", categorical_part, categorical_cols)])

Chạy thử với bốn cột số và hai cột phân loại để chứng minh cơ chế hoạt động.

In [ ]:
DEMO_NUMERIC = ["loan_amnt", "int_rate", "dti", "annual_inc"]
DEMO_CATEGORICAL = ["grade", "purpose"]
demo_prep = make_preprocessor(DEMO_NUMERIC, DEMO_CATEGORICAL)
demo_prep.fit(train_c)

In [ ]:
demo_train = demo_prep.transform(train_c)
demo_val = demo_prep.transform(val_c)
demo_test = demo_prep.transform(test_c)
print("Kích thước sau biến đổi - train:", demo_train.shape, "| val:", demo_val.shape, "| test:", demo_test.shape)
print("Có giá trị thiếu còn sót lại:", bool(np.isnan(demo_train).any() or np.isnan(demo_val).any() or np.isnan(demo_test).any()))

**Chứng minh pipeline không học gì từ val/test.**
1. Trung vị mà pipeline đã học phải **bằng** trung vị tính tay trên train và **khác** trung vị của val/test.
2. Sau khi biến đổi val và test, các con số đã học phải **không đổi**.

In [ ]:
learned_medians = demo_prep.named_transformers_["num"]["impute"].statistics_.copy()
train_medians = train_c[DEMO_NUMERIC].median().to_numpy()
print("Trung vị pipeline đã học == trung vị tính tay trên train:", np.allclose(learned_medians, train_medians))

In [ ]:
medians_compare = pd.DataFrame({
    "đã học (train)": learned_medians,
    "trung vị val": val_c[DEMO_NUMERIC].median().to_numpy(),
    "trung vị test": test_c[DEMO_NUMERIC].median().to_numpy(),
}, index=DEMO_NUMERIC)
medians_compare

In [ ]:
differs = (~np.isclose(learned_medians, medians_compare["trung vị val"].to_numpy())).sum()
print(f"Số cột có trung vị val khác trung vị đã học: {differs}/{len(DEMO_NUMERIC)} -> pipeline không dùng dữ liệu val.")

In [ ]:
medians_after = demo_prep.named_transformers_["num"]["impute"].statistics_
means_after = demo_prep.named_transformers_["num"]["scale"].mean_
_ = demo_prep.transform(val_c), demo_prep.transform(test_c)
unchanged = np.array_equal(learned_medians, demo_prep.named_transformers_["num"]["impute"].statistics_)
print("Các con số đã học không đổi sau khi biến đổi val và test:", unchanged)
print("Các giá trị của 'grade' mà pipeline học từ train:", list(demo_prep.named_transformers_["cat"].categories_[0]))

In [ ]:
unseen_val = set(val_c["grade"].dropna().unique()) - set(demo_prep.named_transformers_["cat"].categories_[0])
unseen_test = set(test_c["grade"].dropna().unique()) - set(demo_prep.named_transformers_["cat"].categories_[0])
print("Giá trị 'grade' ở val chưa từng thấy ở train:", unseen_val or "không có")
print("Giá trị 'grade' ở test chưa từng thấy ở train:", unseen_test or "không có")

---
# Bước 5 - Chọn, lọc bỏ feature và feature engineering

File gốc có 151 cột nhưng phần lớn không dùng được. Ta gán **mỗi cột vào đúng một nhóm** theo thứ tự các câu hỏi sau, và mỗi nhóm có bằng chứng do code tính trên `train`:

1. Cột nhãn và cột phụ trợ?
2. Cột định danh (mỗi khoản vay một giá trị riêng)?
3. Cột chỉ có sau khi khoản vay đã bắt đầu (rò rỉ thông tin - *leakage*) hoặc không rõ thời điểm?
4. Cột của hồ sơ vay chung?
5. Cột chữ có quá nhiều giá trị khác nhau?
6. Cột (gần như) hằng số?
7. Cột thiếu quá nhiều trong `train`?
8. Còn lại: **cột ứng viên** để giữ.

**Tham khảo cách làm của người khác** (theo tên/mô tả project, chưa đối chiếu nội dung gốc):
- Arnav618, `lending-club-credit-risk` - https://github.com/Arnav618/lending-club-credit-risk : phát hiện leakage bằng thước đo thông tin tương hỗ (mutual information) rồi loại cột; ta dùng ROC-AUC từng cột đơn giản hơn nhưng cùng ý tưởng.
- yanxiali, `Predicting-Default-Clients` - https://github.com/yanxiali/Predicting-Default-Clients-of-Lending-Club-Loans : lọc biến bằng tương quan Pearson.

## 5.1 Hồ sơ của từng cột trên train
Hàm sau tính, cho mỗi cột gốc, các con số dùng để phân nhóm: là cột chữ hay số, tỷ lệ thiếu, số giá trị khác nhau, tỷ lệ giá trị phổ biến nhất.

In [ ]:
def profile_columns(df, columns):
    rows = {}
    for col in columns:
        s = df[col]
        top_share = s.value_counts(normalize=True, dropna=True).iloc[0] if s.notna().any() else np.nan
        rows[col] = {
            "is_text": not pd.api.types.is_numeric_dtype(s),
            "missing_share": s.isna().mean(),
            "n_unique": s.nunique(),
            "unique_share": s.nunique() / max(s.notna().sum(), 1),
            "top_value_share": top_share,
        }
    return pd.DataFrame(rows).T

In [ ]:
profile = profile_columns(train_c, ALL_COLUMNS)
profile = profile.astype({"is_text": bool, "missing_share": float, "n_unique": int, "unique_share": float, "top_value_share": float})
print("Đã tính hồ sơ cho", len(profile), "cột.")

## 5.2 Cột định danh
Một cột định danh có gần như mỗi dòng một giá trị riêng, nên không mang thông tin để dự đoán.

**Bảng 5.** Các cột có tỷ lệ giá trị khác nhau trên 90% hoặc trống hoàn toàn.

In [ ]:
identifier_candidates = profile[(profile["unique_share"] > 0.9) | (profile["missing_share"] == 1)]
identifier_candidates[["is_text", "missing_share", "n_unique", "unique_share"]]

In [ ]:
UNIQUE_SHARE_ID = 0.99
print("Cột có tỷ lệ giá trị khác nhau từ", UNIQUE_SHARE_ID, "trở lên:", profile.index[profile["unique_share"] >= UNIQUE_SHARE_ID].tolist())
print("Cột trống hoàn toàn trong train:", profile.index[profile["missing_share"] == 1].tolist())
print("Quyết định -> các cột trên là mã định danh (hoặc trống), loại bỏ.")

## 5.3 Cột chỉ có sau khi vay, hoặc không rõ thời điểm (leakage)
Mô hình sẽ dùng để duyệt khoản vay **trước khi** cho vay, nên mọi cột chỉ xuất hiện sau khi khoản vay bắt đầu đều không được dùng. Từ điển ở Notebook 1 đã gán giả thuyết cho từng cột. Giờ ta **kiểm chứng bằng code**.

Với mỗi cột bị nghi, ta đo xem *một mình cột đó* dự đoán nhãn tốt đến đâu bằng ROC-AUC trên train (0,5 là vô dụng, 1,0 là hoàn hảo; ta lấy `max(AUC, 1-AUC)` để không phụ thuộc chiều). Với cột chữ hoặc ngày, ta dùng chỉ báo "có giá trị hay không".

In [ ]:
def single_column_auc(df, column):
    series = df[column]
    if pd.api.types.is_numeric_dtype(series):
        values = series.fillna(series.median())
    else:
        values = series.notna().astype(int)        # cột chữ/ngày: chỉ xét "có giá trị hay không"
    if values.nunique() < 2:
        return 0.5
    auc = roc_auc_score(df["default"], values)
    return max(auc, 1 - auc)

In [ ]:
timing = data_dictionary["known_at_application (giả thuyết)"]
suspect_columns = [c for c in timing.index if timing[c] in ("Không", "Không rõ") and c not in ("loan_status", "issue_d")]
application_columns = [c for c in timing.index if timing[c] == "Có"]
print("Số cột nghi là sau khi vay hoặc không rõ:", len(suspect_columns), "| số cột giả thuyết là biết lúc nộp đơn:", len(application_columns))

In [ ]:
suspect_auc = pd.Series({c: single_column_auc(train_c, c) for c in suspect_columns}).sort_values(ascending=False)
application_auc = pd.Series({c: single_column_auc(train_c, c) for c in application_columns
                             if c in train_c.columns and train_c[c].notna().any()})
print("Đã tính AUC từng cột cho", len(suspect_auc), "cột nghi và", len(application_auc), "cột lúc nộp đơn.")

**Bảng 6.** 15 cột nghi có AUC đơn cột cao nhất.

In [ ]:
top_suspect = suspect_auc.head(15).to_frame("AUC đơn cột")
top_suspect["ý nghĩa"] = data_dictionary.loc[top_suspect.index, "meaning"]
top_suspect

In [ ]:
best_application = application_auc.idxmax()
print(f"Cột lúc nộp đơn tốt nhất khi đứng một mình: {best_application} (AUC {application_auc.max():.3f}).")
above = suspect_auc[suspect_auc > application_auc.max()]
print(f"Số cột nghi có AUC cao hơn cột lúc nộp đơn tốt nhất: {len(above)} / {len(suspect_auc)}.")
print("Các cột đó:", above.index.tolist())
flat = suspect_auc[suspect_auc <= 0.51]
print(f"Số cột nghi có AUC gần 0.5 (<= 0.51): {len(flat)}. Ví dụ: {flat.index[:5].tolist()}")
print("Lưu ý -> AUC thấp không chứng minh cột vô tội: các cột này vẫn mô tả khoản vay SAU khi bắt đầu (ví dụ số gốc còn nợ), "
      "nên loại theo logic thời điểm. AUC cao chỉ là bằng chứng bổ sung, không phải điều kiện.")

Với các cột **không rõ thời điểm** (giả thuyết "Không rõ"), ta xem thêm dữ liệu của chính chúng.

**Bảng 7.** Các cột "Không rõ": tỷ lệ giá trị phổ biến nhất, số giá trị khác nhau và AUC đơn cột.

In [ ]:
uncertain = [c for c in suspect_columns if timing[c] == "Không rõ"]
uncertain_table = profile.loc[uncertain, ["missing_share", "n_unique", "top_value_share"]].copy()
uncertain_table["AUC đơn cột"] = suspect_auc[uncertain]
uncertain_table["ý nghĩa"] = data_dictionary.loc[uncertain, "meaning"]
uncertain_table

In [ ]:
same_as_request = (train_c["funded_amnt"] == train_c["loan_amnt"]).mean()
print(f"funded_amnt bằng đúng loan_amnt ở {same_as_request:.1%} số khoản vay trong train: gần như trùng với số tiền người vay xin, nên không thêm thông tin.")
print("Quyết định -> với cột không rõ thời điểm ta chọn an toàn: loại bỏ, để tránh vô tình dùng thông tin biết sau khi cho vay.")

## 5.4 Cột của hồ sơ vay chung
Các cột `*_joint` và `sec_app_*` chỉ có giá trị khi hồ sơ là vay chung. **Bảng 8.** Tỷ lệ thiếu của các cột này theo `application_type` (trên train).

In [ ]:
joint_columns = [c for c in ALL_COLUMNS if c.endswith("_joint") or c.startswith("sec_app_")]
missing_by_type = train_c.groupby("application_type")[joint_columns].apply(lambda g: g.isna().mean()).T * 100
missing_by_type.columns = [f"thiếu % ({c})" for c in missing_by_type.columns]
missing_by_type

In [ ]:
type_share = train_c["application_type"].value_counts(normalize=True)
print("Tỷ lệ theo application_type trong train:", {k: f"{v:.1%}" for k, v in type_share.items()})
print(f"Số cột hồ sơ vay chung: {len(joint_columns)}; tỷ lệ thiếu trung bình toàn train: {train_c[joint_columns].isna().mean().mean():.1%}.")
print("Quyết định -> các cột này gần như trống với hồ sơ cá nhân (chiếm đa số), loại bỏ; thông tin vay chung vẫn còn trong cột application_type.")

## 5.5 Cột chữ có quá nhiều giá trị khác nhau
Cột chữ có hàng trăm giá trị (tên công việc, mô tả tự do...) không dùng trực tiếp được, trừ khi là ngày tháng sẽ đổi thành số. Ta tìm ngưỡng từ phân phối số giá trị khác nhau của các cột chữ.

**Bảng 9.** Các cột chữ sắp theo số giá trị khác nhau trong train.

In [ ]:
text_profile = profile[profile["is_text"]].sort_values("n_unique", ascending=False)
text_profile[["n_unique", "missing_share"]].assign(ý_nghĩa=data_dictionary.loc[text_profile.index, "meaning"]).head(20)

In [ ]:
def is_month_date(series, sample=2000):
    values = series.dropna().head(sample)
    if values.empty:
        return False
    return pd.to_datetime(values, format="%b-%Y", errors="coerce").notna().mean() > 0.99

In [ ]:
text_columns = profile.index[profile["is_text"]]
date_like = [c for c in text_columns if is_month_date(train_c[c])]
print("Cột chữ có dạng ngày tháng (vd 'Dec-2015'):", date_like)

In [ ]:
MAX_UNIQUE_CATEGORY = 100
many_values = [c for c in text_columns if profile.loc[c, "n_unique"] > MAX_UNIQUE_CATEGORY and c not in date_like]
print(f"Cột chữ (không phải ngày) có hơn {MAX_UNIQUE_CATEGORY} giá trị khác nhau:", many_values)
few_values = profile.loc[[c for c in text_columns if c not in date_like and c not in many_values], "n_unique"].sort_values()
print("Số giá trị khác nhau lớn nhất của các cột chữ còn lại:", few_values.index[-1], int(few_values.iloc[-1]))
print("Quyết định -> loại các cột nhiều giá trị (one-hot sẽ tạo quá nhiều cột); giữ cột ngày để đổi sang số ở mục 5.9.")

## 5.6 Cột (gần như) hằng số
Cột mà một giá trị chiếm gần hết các dòng gần như không giúp phân biệt khoản vay.

**Bảng 10.** Các cột có giá trị phổ biến nhất chiếm từ 95% trở lên (trên train).

In [ ]:
near_constant_view = profile[profile["top_value_share"] >= 0.95][["top_value_share", "missing_share", "n_unique"]]
near_constant_view = near_constant_view.assign(ý_nghĩa=data_dictionary.loc[near_constant_view.index, "meaning"])
near_constant_view.sort_values("top_value_share", ascending=False)

In [ ]:
TOP_SHARE_CONSTANT = 0.99
constant_columns = profile.index[profile["top_value_share"] >= TOP_SHARE_CONSTANT].tolist()
print(f"Số cột có một giá trị chiếm từ {TOP_SHARE_CONSTANT:.0%} dòng trở lên: {len(constant_columns)}")
borderline = profile[(profile["top_value_share"] >= 0.95) & (profile["top_value_share"] < TOP_SHARE_CONSTANT)]
print("Cột sát ngưỡng (95%-99%), được giữ lại:", borderline.index.tolist())

## 5.7 Cột thiếu quá nhiều trong train
Ngưỡng thiếu được tính **chỉ trên train**. Để chọn ngưỡng hợp lý, ta xem phân phối tỷ lệ thiếu của các cột còn lại và năm đầu tiên mỗi cột có dữ liệu.

**Bảng 11.** Số cột theo khoảng tỷ lệ thiếu trong train.

In [ ]:
missing_bins = pd.cut(profile["missing_share"], [-0.001, 0, 0.05, 0.10, 0.30, 0.60, 0.90, 1.0],
                      labels=["0%", "0-5%", "5-10%", "10-30%", "30-60%", "60-90%", "90-100%"])
missing_bins.value_counts().sort_index().to_frame("số cột")

In [ ]:
MAX_MISSING = 0.30
too_empty = profile.index[profile["missing_share"] > MAX_MISSING].tolist()
mid_missing = profile[(profile["missing_share"] > 0.05) & (profile["missing_share"] <= MAX_MISSING)]
print(f"Số cột thiếu trên {MAX_MISSING:.0%} trong train: {len(too_empty)}")
print(f"Số cột thiếu từ 5% đến {MAX_MISSING:.0%} (được giữ và điền trung vị): {len(mid_missing)}")

In [ ]:
first_year_train = {c: train_c.loc[train_c[c].notna(), "year"].min() for c in mid_missing.index}
print("Năm đầu tiên có dữ liệu của các cột thiếu 5%-30% (trong train):")
print(pd.Series(first_year_train).value_counts().sort_index().to_string())
print("Lưu ý -> các cột này thiếu vì được Lending Club bắt đầu ghi từ năm đó; điền trung vị cho các khoản vay trước đó là một giả định đơn giản, "
      "ghi lại như một hạn chế.")

## 5.8 Gán mỗi cột vào đúng một nhóm
Hàm `classify_column` đi qua các câu hỏi theo đúng thứ tự ở đầu bước 5 và trả về nhóm của cột ngay khi gặp câu trả lời "có" đầu tiên. Vì dùng thứ tự cố định nên mỗi cột chỉ rơi vào một nhóm.

In [ ]:
META_COLUMNS = ["default", "loan_status", "issue_d", "issue_date", "year", "resolved", "matured"]

In [ ]:
def classify_column(column):
    p = profile.loc[column] if column in profile.index else None
    if column in META_COLUMNS or p is None:
        return "nhãn / cột phụ trợ"
    if p["unique_share"] >= UNIQUE_SHARE_ID or p["missing_share"] == 1:
        return "định danh / trống hoàn toàn"
    if timing[column] in ("Không", "Không rõ"):
        return "sau khi vay / không rõ thời điểm"
    if column in joint_columns:
        return "hồ sơ vay chung"
    if column in many_values:
        return "chữ có quá nhiều giá trị"
    if p["top_value_share"] >= TOP_SHARE_CONSTANT:
        return "(gần như) hằng số"
    if p["missing_share"] > MAX_MISSING:
        return "thiếu quá nhiều trong train"
    return "giữ lại làm ứng viên"

In [ ]:
column_group = pd.Series({c: classify_column(c) for c in ALL_COLUMNS}, name="nhóm")
print("Đã phân nhóm", len(column_group), "cột.")

In [ ]:
assert len(column_group) == len(ALL_COLUMNS) == 151
assert set(column_group.index) == set(ALL_COLUMNS)
assert column_group.notna().all()
print("Kiểm tra qua: cả 151 cột đều được gán đúng một nhóm, không sót cột nào.")

**Bảng 12.** Số cột trong từng nhóm.

In [ ]:
group_summary = column_group.value_counts().to_frame("số cột")
group_summary.loc["Tổng"] = group_summary["số cột"].sum()
group_summary

In [ ]:
kept_candidates = column_group[column_group == "giữ lại làm ứng viên"].index.tolist()
print(f"{len(ALL_COLUMNS)} cột gốc -> {len(kept_candidates)} cột ứng viên được giữ lại ({len(kept_candidates) / len(ALL_COLUMNS):.0%}).")

**Bảng 13.** Các cột ứng viên được giữ lại, kèm ý nghĩa và kiểu dữ liệu.

In [ ]:
kept_table = data_dictionary.loc[kept_candidates, ["meaning"]].assign(
    kiểu=["chữ" if profile.loc[c, "is_text"] else "số" for c in kept_candidates],
    thiếu_pct=profile.loc[kept_candidates, "missing_share"] * 100)
kept_table

In [ ]:
dropped_leak = column_group[column_group == "sau khi vay / không rõ thời điểm"].index
used_by_name = [c for c in dropped_leak if c in kept_candidates]
assert not used_by_name
print("Kiểm tra qua: không cột nào thuộc nhóm 'sau khi vay / không rõ thời điểm' nằm trong danh sách giữ lại.")

## 5.9 Feature engineering
Từ các cột ứng viên, ta tạo thêm feature dễ dùng hơn. Mỗi feature là **một hàm riêng**, là công thức cố định theo từng dòng (không học gì từ dữ liệu), nên áp dụng cho cả ba tập được.

| Feature mới | Công thức | Lý do |
|---|---|---|
| `term_months` | chữ `" 36 months"` -> số 36 | Mô hình cần số |
| `emp_length_years` | `"< 1 year"` -> 0, `"10+ years"` -> 10 | Mô hình cần số |
| `fico_avg` | trung bình `fico_range_low` và `fico_range_high` | Hai cột này gần như cùng một thông tin |
| `credit_history_years` | (ngày giải ngân - ngày mở tài khoản đầu tiên) / 365.25 | Lịch sử tín dụng dài hay ngắn |
| `loan_to_income` | `loan_amnt / annual_inc` | Khoản vay lớn so với thu nhập |
| `log_annual_inc`, `log_revol_bal` | log(1 + x) | Thu nhập và dư nợ lệch phải mạnh |

In [ ]:
def make_term_months(df):
    return df["term"].str.extract(r"(\d+)")[0].astype(float)

In [ ]:
def make_emp_length_years(df):
    text = df["emp_length"].replace({"< 1 year": "0 years"})
    return text.str.extract(r"(\d+)")[0].astype(float)

In [ ]:
def make_fico_avg(df):
    return (df["fico_range_low"] + df["fico_range_high"]) / 2

In [ ]:
def make_credit_history_years(df):
    first_credit = parse_month(df["earliest_cr_line"])
    return (df["issue_date"] - first_credit).dt.days / 365.25

Tỷ lệ `loan_to_income` có thể rất lớn nếu thu nhập khai quá nhỏ. Ta chặn trên bằng phân vị 99.9% của **train** (một con số cố định học từ train, áp dụng như nhau cho cả ba tập).

In [ ]:
ratio_train = train_c["loan_amnt"] / train_c["annual_inc"]
LOAN_TO_INCOME_CAP = round(float(ratio_train.quantile(0.999)), 3)
print("LOAN_TO_INCOME_CAP (phân vị 99.9% của train) =", LOAN_TO_INCOME_CAP, "| giá trị lớn nhất trong train:", round(float(ratio_train.max()), 3))

In [ ]:
def make_loan_to_income(df):
    return (df["loan_amnt"] / df["annual_inc"]).clip(upper=LOAN_TO_INCOME_CAP)

In [ ]:
def make_log_annual_inc(df):
    return np.log1p(df["annual_inc"])

In [ ]:
def make_log_revol_bal(df):
    return np.log1p(df["revol_bal"])

Hàm `add_features` ghép tất cả các hàm trên và thêm vào bảng dữ liệu.

In [ ]:
def add_features(df):
    df = df.copy()
    df["term_months"] = make_term_months(df)
    df["emp_length_years"] = make_emp_length_years(df)
    df["fico_avg"] = make_fico_avg(df)
    df["credit_history_years"] = make_credit_history_years(df)
    df["loan_to_income"] = make_loan_to_income(df)
    df["log_annual_inc"] = make_log_annual_inc(df)
    df["log_revol_bal"] = make_log_revol_bal(df)
    return df

In [ ]:
NEW_FEATURES = ["term_months", "emp_length_years", "fico_avg", "credit_history_years",
                "loan_to_income", "log_annual_inc", "log_revol_bal"]
needed_raw = ["term", "emp_length", "fico_range_low", "fico_range_high", "earliest_cr_line",
              "loan_amnt", "annual_inc", "revol_bal"]
print("Cột gốc mà các hàm feature cần nhưng không nằm trong danh sách giữ lại:", [c for c in needed_raw if c not in kept_candidates])

In [ ]:
assert all(c in kept_candidates for c in needed_raw)
train_f = add_features(train_c)
val_f = add_features(val_c)
test_f = add_features(test_c)
print("Đã thêm feature cho cả ba tập. Số cột train_f:", train_f.shape[1])

### Kiểm tra từng feature mới trên train
Ba câu hỏi: (1) chuyển đổi có đúng ý không (in vài dòng gốc cạnh dòng mới)? (2) feature mới có thiếu nhiều không? (3) AUC đơn cột có tốt hơn cột gốc không?

**Bảng 14.** Đối chiếu giá trị gốc và giá trị sau chuyển đổi.

In [ ]:
print("emp_length -> emp_length_years")
print(train_f.groupby(train_f["emp_length"].fillna("(thiếu)"))["emp_length_years"].first().to_string())
print()
print("term -> term_months:", train_f.groupby("term")["term_months"].first().to_dict())

In [ ]:
compare = train_f[["fico_range_low", "fico_range_high", "fico_avg", "loan_amnt", "annual_inc", "loan_to_income",
                   "earliest_cr_line", "issue_date", "credit_history_years"]].head(5)
compare

In [ ]:
print("Tỷ lệ thiếu (%) của các feature mới trong train:")
print((train_f[NEW_FEATURES].isna().mean() * 100).round(2).to_string())
print("Lịch sử tín dụng nhỏ nhất:", round(train_f["credit_history_years"].min(), 2), "năm (phải dương nếu logic đúng).")

**Bảng 15.** ROC-AUC đơn cột của feature mới so với cột gốc (train).

In [ ]:
pairs = [("fico_range_low", "fico_avg"), ("annual_inc", "log_annual_inc"), ("revol_bal", "log_revol_bal"),
         ("loan_amnt", "loan_to_income"), ("term", "term_months"), ("emp_length", "emp_length_years"),
         ("earliest_cr_line", "credit_history_years")]
rows = []
for raw_col, new_col in pairs:
    raw_auc = single_column_auc(train_f, raw_col) if pd.api.types.is_numeric_dtype(train_f[raw_col]) else np.nan
    rows.append((raw_col, raw_auc, new_col, single_column_auc(train_f, new_col)))
auc_pairs = pd.DataFrame(rows, columns=["cột gốc", "AUC gốc", "feature mới", "AUC mới"]).set_index("feature mới")
auc_pairs

In [ ]:
for new_col, row in auc_pairs.iterrows():
    if pd.isna(row["AUC gốc"]):
        print(f"{new_col}: AUC {row['AUC mới']:.3f} (cột gốc {row['cột gốc']} là chữ, không đo AUC trực tiếp được -> đây là cách duy nhất để dùng cột này)")
    else:
        delta = row["AUC mới"] - row["AUC gốc"]
        verdict = "tốt hơn" if delta > 0.001 else ("như nhau" if abs(delta) <= 0.001 else "kém hơn")
        print(f"{new_col}: AUC {row['AUC mới']:.3f} so với {row['cột gốc']} {row['AUC gốc']:.3f} -> {verdict}")
print("Ghi chú -> hàm log giữ nguyên thứ tự các giá trị nên không đổi AUC đơn cột; ta vẫn giữ vì giúp các mô hình tuyến tính xử lý phân phối lệch.")

## 5.10 Lọc các cột trùng thông tin bằng tương quan
Hai cột số có tương quan Pearson rất cao mang gần như cùng thông tin. Ta tìm các cặp có |r| lớn trên **train** và với mỗi cặp giữ cột có AUC đơn cột cao hơn.

Trước hết lập danh sách cột số ứng viên: cột số được giữ lại, cộng feature mới, trừ các cột gốc đã được thay thế bởi feature mới.

In [ ]:
replaced_by_new = ["fico_range_low", "fico_range_high", "annual_inc", "revol_bal"]
numeric_candidates = [c for c in kept_candidates if not profile.loc[c, "is_text"] and c not in replaced_by_new]
numeric_candidates += ["term_months", "emp_length_years", "fico_avg", "credit_history_years",
                       "loan_to_income", "log_annual_inc", "log_revol_bal"]
print("Số cột số ứng viên:", len(numeric_candidates))

In [ ]:
corr = train_f[numeric_candidates].corr()
print("Đã tính ma trận tương quan:", corr.shape)

In [ ]:
def high_corr_pairs(corr_matrix, threshold):
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)).stack().dropna()
    return upper[upper.abs() >= threshold].sort_values(key=abs, ascending=False)

In [ ]:
CORR_THRESHOLD = 0.90
pairs_high = high_corr_pairs(corr, CORR_THRESHOLD)
print(f"Số cặp cột có |tương quan| >= {CORR_THRESHOLD}: {len(pairs_high)}")

**Bảng 16.** Các cặp cột tương quan cao nhất và cột được chọn giữ (cột có AUC đơn cột cao hơn).

In [ ]:
auc_numeric = pd.Series({c: single_column_auc(train_f, c) for c in numeric_candidates})
rows = []
for (a, b), r in pairs_high.items():
    keep, drop = (a, b) if auc_numeric[a] >= auc_numeric[b] else (b, a)
    rows.append({"cột A": a, "cột B": b, "tương quan": r, "AUC A": auc_numeric[a], "AUC B": auc_numeric[b], "bỏ": drop})
pair_table = pd.DataFrame(rows)
pair_table

In [ ]:
drop_by_corr = sorted(set(pair_table["bỏ"])) if len(pair_table) else []
print(f"Các cột bị bỏ vì trùng thông tin: {drop_by_corr}")
NUMERIC_FINAL = [c for c in numeric_candidates if c not in drop_by_corr]
print(f"Cột số cuối cùng: {len(NUMERIC_FINAL)} (từ {len(numeric_candidates)} ứng viên).")

## 5.11 Danh sách feature cuối cùng
Cột phân loại: các cột chữ ứng viên (không phải ngày tháng) mà ta chưa thay bằng feature số.

In [ ]:
replaced_text = ["term", "emp_length", "earliest_cr_line"]
CATEGORICAL_FINAL = [c for c in kept_candidates if profile.loc[c, "is_text"] and c not in replaced_text]
print("Cột phân loại cuối cùng:", CATEGORICAL_FINAL)
print("Số giá trị khác nhau của từng cột phân loại:", {c: int(profile.loc[c, "n_unique"]) for c in CATEGORICAL_FINAL})

In [ ]:
assert not set(NUMERIC_FINAL) & set(CATEGORICAL_FINAL)
for bad in ["default", "loan_status", "id", "issue_date"]:
    assert bad not in NUMERIC_FINAL + CATEGORICAL_FINAL
print("Kiểm tra qua: nhãn, id và ngày giải ngân không nằm trong danh sách feature.")
print(f"Tổng số feature đầu vào cho mô hình: {len(NUMERIC_FINAL)} số + {len(CATEGORICAL_FINAL)} phân loại.")

## 5.12 Ghép thành một pipeline hoàn chỉnh
Pipeline ghép ba bước: làm sạch giá trị bất thường, tạo feature mới, rồi điền thiếu / chuẩn hóa / one-hot. Pipeline nhận dữ liệu **thô** (các cột gốc được giữ lại) và trả ra ma trận số. Ta `fit` chỉ trên `train`.

In [ ]:
INPUT_COLUMNS = sorted(set(kept_candidates) | {"issue_date"})
print("Số cột đầu vào thô của pipeline:", len(INPUT_COLUMNS))

In [ ]:
full_pipeline = Pipeline([
    ("clean", FunctionTransformer(fix_invalid_values)),
    ("features", FunctionTransformer(add_features)),
    ("prep", make_preprocessor(NUMERIC_FINAL, CATEGORICAL_FINAL)),
])

In [ ]:
y_train, y_val, y_test = train["default"].to_numpy(), val["default"].to_numpy(), test["default"].to_numpy()
X_train_raw, X_val_raw, X_test_raw = train[INPUT_COLUMNS], val[INPUT_COLUMNS], test[INPUT_COLUMNS]
full_pipeline.fit(X_train_raw)
print("Đã fit pipeline trên train.")

In [ ]:
X_train = full_pipeline.transform(X_train_raw)
X_val = full_pipeline.transform(X_val_raw)
X_test = full_pipeline.transform(X_test_raw)
print("Kích thước ma trận - train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

In [ ]:
feature_names = list(full_pipeline.named_steps["prep"].get_feature_names_out())
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == len(feature_names)
print("Số cột sau biến đổi (gồm cả cột one-hot):", len(feature_names))
print("Có giá trị thiếu còn sót:", bool(np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any()))

**Bảng 17.** Thống kê của ma trận train sau biến đổi (cột số đã chuẩn hóa nên trung bình gần 0, độ lệch chuẩn gần 1).

In [ ]:
numeric_block = pd.DataFrame(X_train[:, :len(NUMERIC_FINAL)], columns=NUMERIC_FINAL)
numeric_block.describe().T[["mean", "std", "min", "max"]]

In [ ]:
print(f"Trung bình các cột số train sau chuẩn hóa: lớn nhất theo trị tuyệt đối {numeric_block.mean().abs().max():.2e}.")
val_block = pd.DataFrame(X_val[:, :len(NUMERIC_FINAL)], columns=NUMERIC_FINAL)
print(f"Trung bình cột số ở val: lớn nhất theo trị tuyệt đối {val_block.mean().abs().max():.3f} "
      "(không về 0 vì val dùng trung bình/độ lệch chuẩn của train - đúng như thiết kế).")

**Kiểm tra pipeline cho kết quả giống hệt khi chạy lại** (không có yếu tố ngẫu nhiên).

In [ ]:
pipeline_again = Pipeline([
    ("clean", FunctionTransformer(fix_invalid_values)),
    ("features", FunctionTransformer(add_features)),
    ("prep", make_preprocessor(NUMERIC_FINAL, CATEGORICAL_FINAL)),
]).fit(X_train_raw)
print("Fit lần hai cho ma trận val giống hệt lần một:", np.allclose(pipeline_again.transform(X_val_raw), X_val))

## 5.13 Lưu pipeline và danh sách feature cho bước mô hình hóa

In [ ]:
joblib.dump(full_pipeline, ART_DIR / "preprocess_pipeline.joblib")
feature_info = {"numeric": NUMERIC_FINAL, "categorical": CATEGORICAL_FINAL, "input_columns": INPUT_COLUMNS,
                "thresholds": {"DTI_MAX": DTI_MAX, "REVOL_UTIL_CAP": REVOL_UTIL_CAP, "LOAN_TO_INCOME_CAP": LOAN_TO_INCOME_CAP,
                               "MAX_MISSING": MAX_MISSING, "TOP_SHARE_CONSTANT": TOP_SHARE_CONSTANT,
                               "MAX_UNIQUE_CATEGORY": MAX_UNIQUE_CATEGORY, "CORR_THRESHOLD": CORR_THRESHOLD}}
(ART_DIR / "feature_info.json").write_text(json.dumps(feature_info, indent=2, ensure_ascii=False))
column_group.to_frame().join(data_dictionary[["meaning"]]).to_csv(ART_DIR / "column_groups.csv", encoding="utf-8-sig")
print("Đã lưu preprocess_pipeline.joblib, feature_info.json, column_groups.csv vào", ART_DIR.resolve())

In [ ]:
print("Tóm tắt Bước 4 - 5:")
print(f"  - {len(ALL_COLUMNS)} cột gốc -> {len(kept_candidates)} cột ứng viên -> {len(NUMERIC_FINAL)} số + {len(CATEGORICAL_FINAL)} phân loại cuối cùng")
print(f"  - ma trận sau biến đổi: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")
print("  - mọi thứ học từ dữ liệu (trung vị, chuẩn hóa, danh sách giá trị, các ngưỡng) chỉ tính trên train")